In [1]:
%pip install -U -q pip setuptools wheel setuptools-scm
%pip uninstall -y -q diffusers
%pip install -q transformers==4.44.2 accelerate bitsandbytes seqeval scikit-learn
%pip install -q -U transformers tokenizers accelerate
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... d

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("hf: ok")

hf: ok


In [3]:
import json, sys, importlib, random, re
import numpy as np


SEED = 42
random.seed(SEED); np.random.seed(SEED)

ENTITY_TYPES = ["ITEM", "QTY", "UNIT", "VARIANT", "ANAPHORIC"]
LABELS = ["O"] + [f"{p}-{e}" for e in ENTITY_TYPES for p in ("B", "I")]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

DATA = "/kaggle/input/datasets/sebastianabe/nyatet-order-train-b5"
GENERATE_DATA = "/kaggle/input/datasets/sebastianabe/nyatet-order-train"

sys.path.insert(0, GENERATE_DATA)
import generate_data; importlib.reload(generate_data)

pool = generate_data.generate_dataset(n_orders=2000)   
eval_rows = json.load(open(f"{DATA}/eval_annotated.json", encoding="utf-8"))

print(f"few-shot pool {len(pool)} | eval {len(eval_rows)}")
print(f"  eval with spans {sum(1 for r in eval_rows if r['spans'])}"
      f" | negatives {sum(1 for r in eval_rows if not r['spans'])}")

few-shot pool 2700 | eval 81
  eval with spans 40 | negatives 41


In [4]:
print(len(eval_rows))
print("with spans:", sum(1 for r in eval_rows if r["spans"]))
print("negatives:", sum(1 for r in eval_rows if not r["spans"]))

81
with spans: 40
negatives: 41


In [5]:
SYSTEM = """Anda adalah sistem ekstraksi entitas untuk pesan pemesanan berbahasa Indonesia informal (dengan campuran bahasa Banjar).

Tugas: temukan span berikut di dalam pesan, PERSIS seperti tertulis.

- ITEM      : nama produk. contoh: risol, risoles, resol, bronies
- QTY       : jumlah. contoh: 20, 10, dua, 320
- UNIT      : satuan. contoh: biji, buting, kotak, pcs, loyang
- VARIANT   : keadaan olahan. contoh: mentah, digoreng, frozen, sdh masak
- ANAPHORIC : rujukan ke pesanan sebelumnya. contoh: ky biasa, kaya kmrn

Aturan:
1. Teks span harus SALINAN PERSIS dari pesan. Jangan mengubah ejaan.
2. Jangan menandai waktu (jam 7, jam 08.15) atau harga (55 ribu) sebagai QTY.
3. Banyak pesan bukan pesanan sama sekali. Jika tidak ada entitas, kembalikan [].
4. Kembalikan HANYA JSON, tanpa penjelasan, tanpa markdown.

Format: [{"type": "ITEM", "text": "risol"}, {"type": "QTY", "text": "20"}]"""

def format_example(row):
    spans = [{"type": s["type"], "text": row["text"][s["start"]:s["end"]]}
             for s in row["spans"]]
    return f'Pesan: {row["text"]}\nJawab: {json.dumps(spans, ensure_ascii=False)}'

print(SYSTEM[:400], "...\n")
print(format_example(eval_rows[0]))

Anda adalah sistem ekstraksi entitas untuk pesan pemesanan berbahasa Indonesia informal (dengan campuran bahasa Banjar).

Tugas: temukan span berikut di dalam pesan, PERSIS seperti tertulis.

- ITEM      : nama produk. contoh: risol, risoles, resol, bronies
- QTY       : jumlah. contoh: 20, 10, dua, 320
- UNIT      : satuan. contoh: biji, buting, kotak, pcs, loyang
- VARIANT   : keadaan olahan. co ...

Pesan: Pesan hari ni risol 20 adakah Bu, kemarin Ulun kewarung Pian kd bejualan☺️
Jawab: [{"type": "ITEM", "text": "risol"}, {"type": "QTY", "text": "20"}]


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

K = 8

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
pool_mat = vec.fit_transform([r["text"] for r in pool])

def retrieve(text, k=K):
    sims = cosine_similarity(vec.transform([text]), pool_mat)[0]
    return [pool[i] for i in sims.argsort()[-k:][::-1]]

demo = eval_rows[0]
print(f"query: {demo['text']!r}\n")
for r in retrieve(demo["text"], 3):
    print("  ", r["text"][:60])

query: 'Pesan hari ni risol 20 adakah Bu, kemarin Ulun kewarung Pian kd bejualan☺️'

   ulun pesan 5 buting hari ini adakh
   ulun pesan begoreng 1 hari ini adakh
   ulun pesan risoles masak 10 kotak hari ini adakh


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BNB = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=BNB, device_map="auto")
    m.eval()
    return tok, m

MODELS = {
    "SEA-LION v4 8B": "aisingapore/Apertus-SEA-LION-v4-8B-IT",
        "Sahabat-AI 8B": "GoToCompany/llama3-8b-cpt-sahabatai-v1-instruct",
}

In [8]:
JSON_BLOCK = re.compile(r"\[.*?\]", re.S)

def extract(tok, model, text, k=K, max_new_tokens=200):
    shots = "\n\n".join(format_example(r) for r in retrieve(text, k))
    prompt = f"{SYSTEM}\n\n{shots}\n\nPesan: {text}\nJawab:"

    msgs = [{"role": "user", "content": prompt}]
    try:
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    except Exception:
        ids = tok(prompt, return_tensors="pt")
    if not torch.is_tensor(ids):
        ids = ids["input_ids"]
    ids = ids.to(model.device)

    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens,
                             do_sample=False,
                             pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

    cleaned = raw.split("assistant")[0]
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()
    m = JSON_BLOCK.search(cleaned)
    if not m:
        return [], raw, False
    try:
        parsed = json.loads(m.group(0))
    except json.JSONDecodeError:
        return [], raw, False
    if not isinstance(parsed, list):
        return [], raw, False

    spans = [s for s in parsed
             if isinstance(s, dict) and s.get("type") in ENTITY_TYPES and isinstance(s.get("text"), str)]
    return spans, raw, True

In [9]:
from seqeval.metrics import f1_score, classification_report

def to_bio(text, spans, tok_words=None):
    words, offsets, pos = [], [], 0
    for w in text.split():
        i = text.index(w, pos)
        words.append(w); offsets.append((i, i + len(w))); pos = i + len(w)

    tags = ["O"] * len(words)
    unlocatable = 0
    for s in spans:
        typ = s["type"]
        if "start" in s and "end" in s:          
            a, b = s["start"], s["end"]
        else:                                     
            frag = s.get("text", "")
            idx = text.lower().find(frag.lower()) if frag else -1
            if idx == -1:
                unlocatable += 1
                continue
            a, b = idx, idx + len(frag)
        entered = False
        for j, (ws, we) in enumerate(offsets):
            if ws < b and we > a:
                tags[j] = f'{"B" if not entered else "I"}-{typ}'
                entered = True
    return tags, unlocatable

def evaluate(name, tok, model, rows):
    gold_seqs, pred_seqs = [], []
    schema_fail = halluc = 0
    fp_negatives = 0
    import time; t0 = time.perf_counter()

    for i, row in enumerate(rows):
        g, _ = to_bio(row["text"], row["spans"])
        spans, raw, ok = extract(tok, model, row["text"])
        if not ok:
            schema_fail += 1
        p, un = to_bio(row["text"], spans)
        halluc += un
        if not row["spans"] and any(t != "O" for t in p):
            fp_negatives += 1
        gold_seqs.append(g); pred_seqs.append(p)
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(rows)}")

    elapsed = time.perf_counter() - t0
    n_neg = sum(1 for r in rows if not r["spans"])
    res = {
        "model": name,
        "f1": f1_score(gold_seqs, pred_seqs),
        "schema_failures": schema_fail,
        "hallucinated_spans": halluc,
        "fp_on_negatives": f"{fp_negatives}/{n_neg}",
        "sec_per_message": elapsed / len(rows),
    }
    print(f"\n=== {name} ===")
    for k, v in res.items():
        print(f"  {k:<20} {v}")
    print()
    print(classification_report(gold_seqs, pred_seqs, digits=3))
    return res

In [10]:
results = []
for name, mid in MODELS.items():
    print(f"\n{'='*60}\nloading {name}\n{'='*60}")
    try:
        tok, model = load(mid)
    except Exception as e:
        print(f"SKIPPED — {type(e).__name__}: {str(e)[:200]}")
        continue
    results.append(evaluate(name, tok, model, eval_rows))
    del model; torch.cuda.empty_cache()
    import gc; gc.collect()


loading SEA-LION v4 8B


config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/177k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  20/81
  40/81
  60/81
  80/81

=== Sahabat-AI 8B ===
  model                Sahabat-AI 8B
  f1                   0.8556149732620321
  schema_failures      0
  hallucinated_spans   10
  fp_on_negatives      1/41
  sec_per_message      136.73338572293827

              precision    recall  f1-score   support

   ANAPHORIC      0.333     0.500     0.400         2
        ITEM      1.000     0.857     0.923        21
         QTY      0.966     0.800     0.875        35
        UNIT      0.900     0.857     0.878        21
     VARIANT      0.789     0.789     0.789        19

   micro avg      0.899     0.816     0.856        98
   macro avg      0.798     0.761     0.773        98
weighted avg      0.912     0.816     0.860        98



In [11]:
TAGGER = {
    "model": "Nyatet tagger (shipped)",
    "f1": 0.8365,
    "schema_failures": 0,
    "hallucinated_spans": 0,
    "fp_on_negatives": "5/41",
    "sec_per_message": 0.0216,
}

rows = [TAGGER] + results
print(f"{'model':<26} {'F1':>7} {'schema':>8} {'halluc':>8} {'FP neg':>8} {'sec/msg':>9}")
for r in rows:
    print(f"{r['model']:<26} {r['f1']:>7.4f} {r['schema_failures']:>8} "
          f"{r['hallucinated_spans']:>8} {r['fp_on_negatives']:>8} {r['sec_per_message']:>9.3f}")

if results:
    slowest = max(r["sec_per_message"] for r in results)
    print(f"\nTagger is {slowest/TAGGER['sec_per_message']:.0f}x faster than the slowest baseline.")
print("\nSchema failures and hallucinated spans are structurally impossible for the")
print("tagger: every output token is a pointer into the input.")

model                           F1   schema   halluc   FP neg   sec/msg
Nyatet tagger (shipped)     0.8365        0        0     5/41     0.022
SEA-LION v4 8B              0.7623        9       72    14/41    12.975
Sahabat-AI 8B               0.8556        0       10     1/41   136.733

Tagger is 6330x faster than the slowest baseline.

Schema failures and hallucinated spans are structurally impossible for the
tagger: every output token is a pointer into the input.
